# Example Notebook

*Demonstration file for the cc change format — not part of the course.*

**Topics:**
- Calling a tool from an agent
- Tools with multiple parameters

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 📋 Part 1: Direct Answer vs Tool Call

By default an agent tries to answer from training data.
Instructions let you specify exactly when to reach for a tool instead.

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Return a simulated current weather report for a city."""
    reports = {
        "london": "Overcast, 12°C",
        "paris": "Sunny, 18°C",
        "tokyo": "Light rain, 15°C",
    }
    return reports.get(city.lower(), f"No data for {city}.")


weather_agent = Agent(
    name="WeatherAgent",
    instructions="Use get_weather only when the user asks about current conditions. For general questions answer directly.",
    model=MODEL,
    tools=[get_weather]
)

#### Run the Agent

In [ ]:
result = await Runner.run(weather_agent, input="What's the weather in London right now?")
print(result.final_output)

### 💡 Why This Works

The agent calls the tool because the question requires live data the model cannot answer from training. Instructions sit above tool docstrings in the decision hierarchy — the agent checks instructions first to decide whether a tool is allowed, then reads the docstring to decide which tool to call.

---

## 🔢 Part 2: Tools with Multiple Parameters

Tools can accept more than one argument — the agent extracts all of them from natural language.

In [ ]:
@function_tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another using fixed demo rates."""
    rates = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "GBP"): 0.79}
    rate = rates.get((from_currency.upper(), to_currency.upper()))
    if rate is None:
        return f"No rate available for {from_currency} → {to_currency}."
    return f"{amount} {from_currency} = {amount * rate:.2f} {to_currency}"


currency_agent = Agent(
    name="CurrencyAgent",
    instructions="Use convert_currency when the user asks to convert between currencies.",
    model=MODEL,
    tools=[convert_currency]
)

**Note:** This agent uses fixed demo exchange rates — it does not call a live API. We're focused on parameter extraction here, not data accuracy.

#### Run the Agent

In [ ]:
result = await Runner.run(currency_agent, input="Convert 250 USD to EUR")
print(result.final_output)

### 💡 Why This Works

The agent extracts the three parameters from natural language and maps them to the function signature. If any required parameter is missing the agent will ask before calling the tool.

---

## 💪 Practice Exercises

### Exercise 1: Unit Converter

*Covers: tool definition — parameters and return types*

Build a `convert_units` tool that handles km↔miles, kg↔lbs, and °C↔°F. Wire it into an agent and test three conversions.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Unit Converter
# --------------------------------------------------------------

# TODO: Define convert_units @function_tool

# TODO: Create an agent with this tool

# TODO: Run three test queries

# --- Write your code below this line ---

## 🎯 Key Takeaways

**Tool use:**
- Define *when* to call a tool in the instructions — not just in the docstring
- Parameter names and type hints are the primary signal to the agent — they are the agent's only guide to what values to pass
- The agent can ask for missing parameters if your instructions tell it to

---

## 📍 Next Step

**Notebook XX: [Next lesson]**  

Build on these tool patterns in the next lesson.